In [0]:
# Notebook parameter: the Unity Catalog catalog to use.
# Jobs can override it, so the catalog name is never hard-coded.
dbutils.widgets.text("catalog", "workspace")
CATALOG = dbutils.widgets.get("catalog")

In [0]:
# One schema per medallion layer, with a description stored in Unity Catalog
layers = {
    "credit_bronze": "Raw Home Credit data as ingested",
    "credit_silver": "Typed, cleaned and validated tables",
    "credit_gold": "Business-level feature, segment and score tables",
    "credit_ml": "Models, monitoring and experiment results",
}
# IF NOT EXISTS makes the notebook idempotent: re-running it changes nothing
for schema, comment in layers.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema} COMMENT '{comment}'")

# Managed volumes hold files (not tables); Unity Catalog decides where they are stored
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.credit_bronze.landing")      # raw Kaggle CSVs
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.credit_bronze.checkpoints")  # Auto Loader state
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.credit_ml.artifacts")        # plots and reports

In [0]:
# Files the project needs in the landing volume
EXPECTED = ["application_train.csv", "application_test.csv", "bureau.csv", "bureau_balance.csv",
            "previous_application.csv", "POS_CASH_balance.csv", "installments_payments.csv",
            "credit_card_balance.csv", "HomeCredit_columns_description.csv"]
# File names currently in the volume
present = {f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/credit_bronze/landing")}
# Report anything still missing
print("Faltan:", [f for f in EXPECTED if f not in present] or "nada")